# 5주차 1차시: 벡터 검색 및 재순위화 (Reranking)

| 주제 | 내용 |
|---|---|
| 벡터 검색 | FAISS 유사도 검색, 키워드 재순위화 |
| 재순위화 | BM25 · LLM Pairwise · Hybrid · Listwise |
| 스코어 필터링 | Fixed / Dynamic Threshold |

In [ ]:
# 환경 설정 및 라이브러리 설치
!pip install -q openai langchain langchain-openai langchain-community faiss-cpu \
    rank_bm25 pandas numpy matplotlib gradio python-dotenv tiktoken

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.5/88.5 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 35.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 60.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 55.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.


In [ ]:
import os
import re
import time
import json
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from scipy import stats
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_community.vectorstores import FAISS
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import ContextualCompressionRetriever, EnsembleRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor
# from dotenv import load_dotenv
# load_dotenv()
import os
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
MODEL = "gpt-4o-mini"
llm = ChatOpenAI(model=MODEL)
embeddings_model = OpenAIEmbeddings(model="text-embedding-3-small")

In [ ]:
documents = [
    Document(page_content="트랜스포머는 Self-Attention 메커니즘을 사용하여 시퀀스 데이터를 병렬로 처리하는 딥러닝 아키텍처입니다.", metadata={"id": "d1"}),
    Document(page_content="BERT는 양방향 트랜스포머 인코더로 MLM과 NSP 태스크로 사전학습됩니다.", metadata={"id": "d2"}),
    Document(page_content="GPT는 단방향 트랜스포머 디코더로 다음 토큰 예측 방식으로 학습합니다.", metadata={"id": "d3"}),
    Document(page_content="RAG는 검색 증강 생성 기법으로 외부 지식을 LLM에 결합하여 할루시네이션을 줄입니다.", metadata={"id": "d4"}),
    Document(page_content="벡터 데이터베이스는 임베딩 벡터를 저장하고 유사도 기반 검색을 수행합니다. FAISS, Pinecone 등이 있습니다.", metadata={"id": "d5"}),
    Document(page_content="파인튜닝은 사전학습된 모델을 특정 도메인 데이터로 추가 학습하는 기법입니다. LoRA, QLoRA가 효율적입니다.", metadata={"id": "d6"}),
    Document(page_content="프롬프트 엔지니어링은 LLM에 효과적인 지시를 설계하는 기법입니다. Few-shot, CoT 등이 있습니다.", metadata={"id": "d7"}),
    Document(page_content="토큰화는 텍스트를 모델이 처리할 수 있는 단위로 분할하는 과정입니다. BPE, WordPiece 등이 사용됩니다.", metadata={"id": "d8"}),
]

In [ ]:
vectorstore = FAISS.from_documents(documents, embeddings_model)

In [ ]:
bm25_retriever = BM25Retriever.from_documents(documents, k=5)

In [ ]:
doc_embeddings = {}

def get_embedding(text):
  return np.array(embeddings_model.embed_query(text))

for doc in doc_embeddings:
  doc_embeddings[doc.metadata['id']] = get_embedding()

## 검색 결과 재순위화 (Reranking) 개요

기본 벡터 검색은 임베딩 유사도만 사용 → 정밀도에 한계가 있음.  
재순위화는 초기 검색 결과를 더 정교한 기준으로 다시 정렬하는 과정.

### Bi-encoder vs Cross-encoder

| 방식 | 구조 | 속도 | 정밀도 | 활용 |
|---|---|---|---|---|
| Bi-encoder | query, doc 각각 인코딩 → 코사인 유사도 | 빠름 | 보통 | 초기 검색 (Recall) |
| Cross-encoder | query + doc 합쳐서 인코딩 | 느림 | 높음 | 재순위화 (Precision) |

### 파이프라인

```
Query → [Bi-encoder 벡터 검색] → 후보 Top-K → [Reranking] → 정제 Top-N → [LLM] → 답변
```

### 벡터 검색 및 키워드 재순위화

- `vector_search`: FAISS 유사도 검색으로 Top-K 반환. 거리를 유사도로 변환 `1/(1+score)`
- `keyword_rerank`: 쿼리 키워드 매칭 수 기준 재정렬
- `rank_change`: 재순위화 전후 순위 변동을 DataFrame으로 비교

In [ ]:
def vector_search(query, vectorstore, top_k = 5):
  results = vectorstore.similarity_search_with_score(query, k= top_k)
  return [(doc, 1.0/ (1.0 + score)) for doc, score in results] # 거리기반인 similarity... 결과의 역수를 취해줌

query = "트랜스포머와 버트의 관계"
results = vector_search(query, vectorstore)

In [ ]:
results

[(Document(id='5e0bced5-f3e4-466c-9d0d-f70dc6152657', metadata={'id': 'd1'}, page_content='트랜스포머는 Self-Attention 메커니즘을 사용하여 시퀀스 데이터를 병렬로 처리하는 딥러닝 아키텍처입니다.'),
  np.float32(0.49338043)),
 (Document(id='efdbe980-e913-4751-9c38-3ad3fc8d7973', metadata={'id': 'd2'}, page_content='BERT는 양방향 트랜스포머 인코더로 MLM과 NSP 태스크로 사전학습됩니다.'),
  np.float32(0.41290897)),
 (Document(id='389dfdd1-7cda-4e8e-9ebe-fdd969e54c43', metadata={'id': 'd7'}, page_content='프롬프트 엔지니어링은 LLM에 효과적인 지시를 설계하는 기법입니다. Few-shot, CoT 등이 있습니다.'),
  np.float32(0.38009387)),
 (Document(id='ef783ea8-ab74-4bc9-b1da-d85bad9cf646', metadata={'id': 'd3'}, page_content='GPT는 단방향 트랜스포머 디코더로 다음 토큰 예측 방식으로 학습합니다.'),
  np.float32(0.3715363)),
 (Document(id='a06aafcc-b9ff-422b-ab62-24e1825a969a', metadata={'id': 'd5'}, page_content='벡터 데이터베이스는 임베딩 벡터를 저장하고 유사도 기반 검색을 수행합니다. FAISS, Pinecone 등이 있습니다.'),
  np.float32(0.365963))]

In [ ]:
def keyword_rerank(query, search_results):
  query_terms = set(query.lower().split()) # 현업..? 토큰 kiwie? nlpk..
  reranked = []

  for doc, orig_score in search_results:
    # 쿼리 토큰화: 사용자의 질문(query)을 공백 기준으로 나누고 소문자로 변환하여 고유한 단어 집합(query_terms)을 만듦
    doc_terms = doc.page_content.lower().split()

    # 키워드 매칭: 검색된 각 문서의 내용(page_content)에 쿼리 단어가 몇 번이나 포함되어 있는지 계산
    keyword_hits = sum(1 for t in doc_terms if t in query_terms) # t가 doc_terms와 query_terms 모두에 있다면.. [1,1]

    # 점수 가산: 기존의 벡터 유사도 점수(orig_score)에 키워드 매칭 횟수에 따른 가중치(0.1)를 더해 새로운 점수(new_score)를 산출
    new_score = orig_score + 0.1*keyword_hits

    # 재정렬: 계산된 새 점수를 기준으로 문서들을 내림차순 정렬하여 반환
    reranked.append((doc, new_score, orig_score))

  reranked.sort(key = lambda x:x[1], reverse = True)
  return reranked

In [ ]:
reranked = keyword_rerank(query, results)
reranked

[(Document(id='5e0bced5-f3e4-466c-9d0d-f70dc6152657', metadata={'id': 'd1'}, page_content='트랜스포머는 Self-Attention 메커니즘을 사용하여 시퀀스 데이터를 병렬로 처리하는 딥러닝 아키텍처입니다.'),
  np.float32(0.49338043),
  np.float32(0.49338043)),
 (Document(id='efdbe980-e913-4751-9c38-3ad3fc8d7973', metadata={'id': 'd2'}, page_content='BERT는 양방향 트랜스포머 인코더로 MLM과 NSP 태스크로 사전학습됩니다.'),
  np.float32(0.41290897),
  np.float32(0.41290897)),
 (Document(id='389dfdd1-7cda-4e8e-9ebe-fdd969e54c43', metadata={'id': 'd7'}, page_content='프롬프트 엔지니어링은 LLM에 효과적인 지시를 설계하는 기법입니다. Few-shot, CoT 등이 있습니다.'),
  np.float32(0.38009387),
  np.float32(0.38009387)),
 (Document(id='ef783ea8-ab74-4bc9-b1da-d85bad9cf646', metadata={'id': 'd3'}, page_content='GPT는 단방향 트랜스포머 디코더로 다음 토큰 예측 방식으로 학습합니다.'),
  np.float32(0.3715363),
  np.float32(0.3715363)),
 (Document(id='a06aafcc-b9ff-422b-ab62-24e1825a969a', metadata={'id': 'd5'}, page_content='벡터 데이터베이스는 임베딩 벡터를 저장하고 유사도 기반 검색을 수행합니다. FAISS, Pinecone 등이 있습니다.'),
  np.float32(0.365963),
  np.float32(0.3659

In [ ]:
def rank_change(original_results, reranked_results):

  orig_ranks = {doc_.metadata['id']: i+1 for i, (doc_, _) in enumerate(original_results)}
  new_ranks = {doc_.metadata['id']: i+1 for i, (doc_, _, _) in enumerate(reranked_results)}

  changes = []

  for doc_id in orig_ranks:
    old_r = orig_ranks[doc_id]
    new_r = new_ranks.get(doc_id, -1)

    changes.append({
        'doc_id' : doc_id,
        'before' : old_r,
        'after' : new_r,
        'change' : old_r - new_r
    })

  return pd.DataFrame(changes).sort_values('after')

In [ ]:
df = rank_change(results, reranked)
df

,doc_id,before,after,change
0,d1,1,1,0
1,d2,2,2,0
2,d7,3,3,0
3,d3,4,4,0
4,d5,5,5,0


### BM25 기반 재순위화

TF-IDF를 개선한 키워드 기반 랭킹 알고리즘.

- `k1` (기본 1.5): 단어 빈도 포화 계수. 높을수록 빈도 영향 증가
- `b` (기본 0.75): 문서 길이 정규화 계수. 1에 가까울수록 길이 영향 증가

In [ ]:
# TF-IDF, 많이 나오는것, 흔하지 않은것..
class BM25Reranker:
  def __init__(self, k1 = 1.5, b=0.75):
    self.k1 = k1
    self.b = b

  def rerank(self, query, search_results): # re-rank 개념 자체는 쿼리 + 검색결과로 다시 re-rank하는거임
    docs = [doc for doc, _ in search_results]
    tokenized = [doc.page_content.lower().split() for doc in docs]

    avg_dl = np.mean([len(t) for t in tokenized])

    df_count = Counter()
    for tokens in tokenized:
      for t in set(tokens):
        df_count[t] += 1
    N = len(docs)

    query_tokens = query.lower().split()
    scored = []

    for i, doc in enumerate(docs):
      score = 0.0
      doc_len = len(tokenized[i])
      tf_count = Counter(tokenized[i])

      for qt in query_tokens:
        tf = tf_count.get(qt, 0)
        if tf == 0:
          continue # 루프를 빠져나가서 상위 루프로 가라

        df = df_count.get(qt, 0)
        idf = math.log((N-df + 0.5) / (df + 0.5) + 1)
        numerator = tf * (self.k1 + 1)
        denominator = tf + self.k1 * (1 - self.b +self.b * doc_len/avg_dl)
        score += idf * numerator / denominator

      scored.append((doc, score))

    scored.sort(key = lambda x:x[1], reverse = True)
    return scored

In [ ]:
bm25_reranker = BM25Reranker()

In [ ]:
bm25_results = bm25_reranker.rerank(query, results)

In [ ]:
bm25_results

[(Document(id='5e0bced5-f3e4-466c-9d0d-f70dc6152657', metadata={'id': 'd1'}, page_content='트랜스포머는 Self-Attention 메커니즘을 사용하여 시퀀스 데이터를 병렬로 처리하는 딥러닝 아키텍처입니다.'),
  0.0),
 (Document(id='efdbe980-e913-4751-9c38-3ad3fc8d7973', metadata={'id': 'd2'}, page_content='BERT는 양방향 트랜스포머 인코더로 MLM과 NSP 태스크로 사전학습됩니다.'),
  0.0),
 (Document(id='389dfdd1-7cda-4e8e-9ebe-fdd969e54c43', metadata={'id': 'd7'}, page_content='프롬프트 엔지니어링은 LLM에 효과적인 지시를 설계하는 기법입니다. Few-shot, CoT 등이 있습니다.'),
  0.0),
 (Document(id='ef783ea8-ab74-4bc9-b1da-d85bad9cf646', metadata={'id': 'd3'}, page_content='GPT는 단방향 트랜스포머 디코더로 다음 토큰 예측 방식으로 학습합니다.'),
  0.0),
 (Document(id='a06aafcc-b9ff-422b-ab62-24e1825a969a', metadata={'id': 'd5'}, page_content='벡터 데이터베이스는 임베딩 벡터를 저장하고 유사도 기반 검색을 수행합니다. FAISS, Pinecone 등이 있습니다.'),
  0.0)]

### LLM 기반 재순위화 (Pairwise Reranking)

LLM에게 문서 목록을 보여주고 관련성 점수(0.0~1.0) 요청.

- 장점: 문서 간 의미적 관계를 LLM이 이해 → 정교한 순위 결정
- 단점: API 비용 발생, 문서 수에 비례해 느려짐
- JSON 응답에서 점수 파싱 후 재정렬

In [ ]:
def llm_rerank(query, search_results, top_k = 3):
  docs_text = '\n'.join(
      f"[{doc.metadata['id']}] {doc.page_content}" for doc, _ in search_results

  )
  # .join()은 리스트 같은 여러 개의 문자열을 하나의 문자열로 합칠 때 사용
    # 예: ", ".join(['사과', '배']) -> "사과, 배"
    # 즉, 리스트 사이사이에 ", "라는 본드를 붙여서 하나로 만드는 것
  score_template = ", ".join(f'"{doc.metadata["id"]}": 0.0-1.0' for doc, _ in search_results)

  scoring_chain = ChatPromptTemplate.from_messages([
      ('system', '당신은 scoring 시스템 입니다. 항상 json 형태로 출력하세요'),
      ('human', """다음 쿼리에 대해 각 문서의 관련성을 0.0~1.0 으로 평가하세요.

      쿼리 : {query}

      문서들:
      {docs_text}

      JSON으로 답하세요:
      {{"scores" : {{ {score_template} }} }}""")

  ]) | llm | StrOutputParser()

  result = scoring_chain.invoke({
      'query' : query,
      'docs_text' : docs_text,
      'score_template' : score_template
  })

  # cleaned = result.strip()
  # parsed = json.loads(cleaned)
  cleaned = result.replace("```json", "").replace("```", "").strip()
  parsed = json.loads(cleaned)
  scores = parsed.get('scores', {})

  scored = []

  for doc, orig in search_results:
    rerank_score = scores.get(doc.metadata['id'], 0.0)
    scored.append((doc, float(rerank_score), orig))

  scored.sort(key = lambda x: x[1], reverse = True)

  return  scored[:top_k]

In [ ]:
llm_results = llm_rerank(query, results, top_k = 3)

In [ ]:
llm_results

[(Document(id='efdbe980-e913-4751-9c38-3ad3fc8d7973', metadata={'id': 'd2'}, page_content='BERT는 양방향 트랜스포머 인코더로 MLM과 NSP 태스크로 사전학습됩니다.'),
  0.9,
  np.float32(0.41290897)),
 (Document(id='ef783ea8-ab74-4bc9-b1da-d85bad9cf646', metadata={'id': 'd3'}, page_content='GPT는 단방향 트랜스포머 디코더로 다음 토큰 예측 방식으로 학습합니다.'),
  0.4,
  np.float32(0.3715363)),
 (Document(id='5e0bced5-f3e4-466c-9d0d-f70dc6152657', metadata={'id': 'd1'}, page_content='트랜스포머는 Self-Attention 메커니즘을 사용하여 시퀀스 데이터를 병렬로 처리하는 딥러닝 아키텍처입니다.'),
  0.3,
  np.float32(0.49338043))]

### 하이브리드 재순위화

BM25 + LLM 점수를 가중 결합. 각 점수를 0~1로 정규화 후 가중 평균.

```
최종 점수 = bm25_weight × BM25_norm + (1 - bm25_weight) × LLM_norm
```

In [ ]:
def hybrid_rerank(query, search_results, bm25_weight = 0.3):
  bm25 = BM25Reranker()
  bm25_scored = bm25.rerank(query, search_results)
  bm25_map = {doc.metadata['id'] : score for doc, score in bm25_scored}

  llm_scored = llm_rerank(query, search_results, top_k = len(search_results))
  llm_map = {doc.metadata['id'] : score for doc, score, _ in llm_scored}

  # 스코어나 metric을 더할때 스케일을 조심해야함..
    # 예를 들면 bm25는 0~100
    # llm score는 0 ~ 10

  def normalize(scores_dict):
    vals =list(scores_dict.values())
    min_, max_ = min(vals), max(vals)
    range_ = max_ - min_ if max_ > min_ else 1e-8
    return {k : (v-min_)/ range_ for k, v in scores_dict.items()}

  bm25_norm = normalize(bm25_map)
  llm_norm = normalize(llm_map)

  combined = []

  for doc, _ in search_results:
    did = doc.metadata['id']
    score = bm25_weight * bm25_norm.get(did, 0) + (1 - bm25_weight) * llm_norm.get(did,0)

    combined.append((doc, score))

  combined.sort(key = lambda x:x[1], reverse = True)

  return combined

In [ ]:
hybrid = hybrid_rerank(query, results)
hybrid


[(Document(id='efdbe980-e913-4751-9c38-3ad3fc8d7973', metadata={'id': 'd2'}, page_content='BERT는 양방향 트랜스포머 인코더로 MLM과 NSP 태스크로 사전학습됩니다.'),
  0.7),
 (Document(id='ef783ea8-ab74-4bc9-b1da-d85bad9cf646', metadata={'id': 'd3'}, page_content='GPT는 단방향 트랜스포머 디코더로 다음 토큰 예측 방식으로 학습합니다.'),
  0.3111111111111111),
 (Document(id='5e0bced5-f3e4-466c-9d0d-f70dc6152657', metadata={'id': 'd1'}, page_content='트랜스포머는 Self-Attention 메커니즘을 사용하여 시퀀스 데이터를 병렬로 처리하는 딥러닝 아키텍처입니다.'),
  0.2333333333333333),
 (Document(id='389dfdd1-7cda-4e8e-9ebe-fdd969e54c43', metadata={'id': 'd7'}, page_content='프롬프트 엔지니어링은 LLM에 효과적인 지시를 설계하는 기법입니다. Few-shot, CoT 등이 있습니다.'),
  0.0),
 (Document(id='a06aafcc-b9ff-422b-ab62-24e1825a969a', metadata={'id': 'd5'}, page_content='벡터 데이터베이스는 임베딩 벡터를 저장하고 유사도 기반 검색을 수행합니다. FAISS, Pinecone 등이 있습니다.'),
  0.0)]

### LLM Listwise 재순위화

| 방식 | 비교 대상 | 특징 |
|---|---|---|
| Pointwise | 문서 1개씩 독립 평가 | 빠르지만 문서 간 관계 무시 |
| Listwise | 문서 목록 전체를 한 번에 평가 | 상대 순위 파악 가능, API 호출 적음 |

LLM에게 순위를 직접 출력하게 하는 방식 (예: "3, 1, 5, 2, 4").

In [ ]:
def llm_listwise_rerank(query, search_results):
  docs_text = '\n'.join(
      f"{i+1}, [{doc.metadata['id']}] {doc.page_content[:80]}" for i, (doc, _) in enumerate(search_results)

  )
  ranking_chain = ChatPromptTemplate.from_messages([
      ('system', "당신은 문서 랭킹 시스템입니다."),

      ('human', """다음 문서들을 쿼리와의 관련성 순서로 정리하세요.

      쿼리 : {query}

      문서들 : {docs_text}

      가장 관련성 높은 순서대로 문서 번호를 쉼표로 나열하세요 (예: 3, 1, 5, 2, 4): """)


  ]) | llm | StrOutputParser()

  result = ranking_chain.invoke({'query': query, 'docs_text':docs_text})

  order =  [int(x.strip()) for x in result.strip().split(',')]

  docs_list = [doc for doc, _ in search_results]
  reranked = []

  for rank, idx in enumerate(order):
    if 1<=idx<=len(docs_list):
      docs = docs_list[idx-1]
      reranked.append((doc, 1.0 - rank*0.1))


  return reranked


In [ ]:
listwise = llm_listwise_rerank(query, results)

AttributeError: 'str' object has no attribute 'metadata'

## 스코어 필터링 (Score Filtering)

재순위화 이후 관련성이 낮은 문서를 제거해 컨텍스트 품질을 높임.

| 전략 | 방식 |
|---|---|
| Fixed Threshold | 고정 점수 이상만 유지 |
| Dynamic Threshold | 평균 - 표준편차 기준 |
| Score Gap | 점수 급락 구간에서 절단 |

In [ ]:
class ScoreFilter:

  def fixed_threshold(scored_docs, threshold = 0.5):
    return {(doc, s) for doc, s in scored_docs if s >= threshold}

  def dynamic_threshold(sored_docs, std_factor):  # mean - ???
    scores = [s for _, s in scored_docs]
    threshold = np.mean(scores) - std_factor * np.std(scores)
    return [(doc, s) for doc, s in scored_docs if s >= threshold]